# Some Standardized shocks to illustrate the main macroeconomic features of the model

This notebook performs a set of standard macroeconomic shocks on a model and displays results.  It can be used to verify the good functioning of the model and also illustrates nicely how to perform simulations on World Bank models in python using the `modelflow` package.

The simulations performed are:

* 5 separate stimulus shocks equal to 1 percent of *ex ante* GDP.  These can be used to compare fiscal multipliers and the impacts of different kinds of stimulus on GDP, potential GDP, consumption and inflation
    * a 1 percent of GDP decrease in indirect taxes
    * a 1 percent of GDP decrease in direct taxes
    * a 1 percent of GDP increase in government spending on goods and services
    * a 1 percent of GDP increase in government spending on investment goods
    * a 1 percent of GDP increase in government spending on transfers to households
* 4 non-fiscal shocks were also run
    * a temporary one-year 1 percent increase in the monetary policy interest rate;
    * a permanent 10 percent depreciation of the currency
    * a permanent one percent increase in total factor productivity;
    * a permanent \$20 increase in the price of crude oil

**Note:**
This Notebook is designed to be run on any World Bank model. To customize the notebook to run on a different model, the string variable (`Cty`), which is defined in section 1.1 will have to be changed to take the value of the Mnemonic of the country to be simulated. Similarly the location of the file containing the model may have to be revised.   While  the World Bank mnemonics are the same across countries, not all countries report all variables.  As a result for some models, some variable names (notably those of shocked variables or the expenditure variables being held constant) may need to be revised. Otherwise the Notebook should run without change on any World Bank model.

In [ ]:
import sys
if sys.platform == 'emscripten':  # only in the browser (JupyterLite): install ModelFlow, see mfsetup.py
    %run mfsetup.py
    await install_modelflow()

## Set up python environment and load model object

To work with modelflow we must first import the python libraries that we wish to work with and then instantiate the model object, which we have chose to call `themodel`;

In [ ]:
from modelclass import model

In [ ]:
#Jupyter notebook code that improves the look of the executed notebook
%load_ext autoreload
%autoreload 2

In [ ]:
#This is code to manage dependencies if the notebook is executed in the Google Colab cloud service
if 'google.colab' in str(get_ipython()):
  import os
  os.system('apt -qqq install graphviz')
  os.system('pip -qqq install ModelFlowIb   ')


In [ ]:
# Models downloaded from the World Bank web site using the model.download_github_repo(reset=1) method of modelflow and
# executing this file from the local version of the file stored on their computer shuold not have to changethe filepath varible

#Create a variable to hold model name (used below to allow re-usability of code across models for different countries)

cty="PAK"
filepath=f'data/{cty}_new'
themodel,bline = model.modelload(filepath,run=True,keep='Baseline',silent=1)

## Prepare the simulations

For each shock a separate dataframe is created. Each of these dataframes is given a name that evokes the shock to be performed. Then each dataframe is modified to reflect the shock that is to be performed.

Following the creation of the dataframes the shocks will be performed and the results stored using the `keep=` syntax of model flow.

### Fiscal policy shocks

If necessary, the two lines below can be uncommented in order to generate a list of all variables in the model that start GGEXP (general government expenditure) and GGREV (general governent revenues) and that end CN (millions of current local currency units).

In [ ]:
print("** Expenditures**")
themodel['???GGEXP*CN'].des #Uncomment to get list of mnemonics and descriptions for government expenditures
print("")
print("** Revenues **")
themodel['???GGREV*CN'].des #Uncomment to get list of mnemonics and descriptions for government revenues

#### Create an expenditures string

The fiscal scenarios below exogenize (hold constant) spending on those elements of government spending that are not being directly shocked.

To facilitate that, the variable `GGepx` is assigned a string containing all of the expenditure variables that are to beheld constant.  This variable is then used when setting up each of the fiscal shocks below. 

This list may need to be adjusted from model to model.

In [ ]:
# Government spending variables to be held constant
GGexp='PAKGGREVOTHRCN PAKGGEXPCAPTCN PAKGGEXPGNFSCN PAKGGEXPTRNSCN PAKGGEXPOTHRCN PAKNEGDIFGOVXN PAKBXFSTREMTCD PAKBMFSTREMTCD'

### The Indirect tax cut

This shock assumes that the main elements of government spending are held constant at their pre-shock levels.  This assumption could be relaxed by commenting out the second line. 

In the model indirect taxes are determined as a function of their lagged effective tax rate (Revenues divided by assumed tax base).  

Inspecting the Goods and services tax revenue equation, one can note that the first term drops out in the forecast period (DUMH=0 in the forecast period) and the equation resolves into an identity where revenues evolve according to the tax base and the effective tax rate of the previous year.

To shock the level of indirect spending revenues in 2025 the add factor is used.  By reducing revenues in 2025 by 1 percent of GDP, the effective tax rate in that year falls.  For subsequent year's it is this lower effective rate that is retained, thereby making the shock permanent.

In [ ]:
themodel['{cty}GGREVGNFSCN'].frml

In [ ]:
fpol_indirect=bline.copy()
fpol_indirect=themodel.fix(bline,f'{GGexp}') # Freeze other spending levels

# GNFS tax revenues are determined by the lagged effective rate.  
# Thus shocking the level of tax revenues in 2025 by one percent of GDP
# will increase the effective tax rate going forward by a constant amount
# consistent with a 1 percent increase in GDP and the tax base for consumption

fpol_indirect=themodel.fix(bline,f'{cty}GGREVGNFSCN')
fpol_indirect=fpol_indirect.mfcalc(
    f'<2025 2050> {cty}GGREVGNFSXN ={cty}GGREVGNFSXN - {cty}NYGDPMKTPCN/(PAKNECONPRVTCN+PAKNECONGOVTCN)')

#solve the model.
tempdf = themodel(fpol_indirect,silent=1,keep=f'1 % of GDP Indirect tax cut')

### DIrect tax hike of 1 % of GDP

The same basic methodology is followed for  direct taxes.


In [ ]:
themodel[f'{cty}GGREVDRCTCN'].frml

In [ ]:
fpol_direct=bline.copy()
fpol_direct=themodel.fix(bline,f'{GGexp}') # Freeze spending levels Capital

# In the model direct taxes (oersonal and corporate income taxes) tax are 
# determined by the lagged effective rate of taxation.  Shocking the 
# level of tax revenues in 2025 by one percent of GDP decreases the effective 
# tax rate going forward by a constant amount consistent with a 1 percent decrease 
# in GDP and the tax base for direct taxes

fpol_direct = fpol_direct.mfcalc('<2025 2050> PAKGGREVDRCTXN = PAKGGREVDRCTXN - 1') #() 100*((PAKGGREVDRCTCN-0.01*PAKNYGDPMKTPCN_0)/((PAKNYYWBTOTLCN-PAKNYYWBINFMCN)+(PAKNYGDPMKTPCN-PAKNYYWBTOTLCN)))') 
#fpol_direct=themodel.fix(fpol_direct,f'{cty}GGREVDRCTCN')
#fpol_direct=fpol_direct.mfcalc(
#    f'<2025 2100> {cty}GGREVDRCTCN_X ={cty}GGREVDDRCTCN_X  - .01*{cty}NYGDPMKTPCN')

#solve the model.
tempdf = themodel(fpol_direct,silent=1,keep=f'1 % of GDP direct tax cut')

### Increase in expenditure on goods and services

The ex ante fiscal effort is the same in this scenario (1% of ex ante GDP) with the difference that it is implemented as an increase government spending, in this instance on goods and services.



In [ ]:
fpol_ExpGS=bline.copy()
fpol_ExpGS=themodel.fix(bline,f'{GGexp}') # Freeze spending levels 

fpol_ExpGS=fpol_ExpGS.mfcalc(
    f'<2025 2050> {cty}GGEXPGNFSCN_X ={cty}GGEXPGNFSCN_X   + .01*{cty}NYGDPMKTPCN')


In [ ]:
#solve the model.
tempdf = themodel(fpol_ExpGS,silent=1,keep=f'1 % of GDP increase in G&S spending')
#themodel.lastdf['IDNNECONGOVTCN']/themodel.basedf['IDNNECONGOVTCN']



### Increase in expenditure on investment goods

The ex ante fiscal effort is the same in this scenario (1% of ex ante GDP), implemented as an increase in government spending on capital goods. 


In [ ]:
fpol_ExpInv=bline.copy()
fpol_ExpInv=themodel.fix(bline,f'{GGexp}') # Freeze spending levels 

fpol_ExpInv=fpol_ExpInv.mfcalc(f'<2025 2050> {cty}GGEXPCAPTCN_X ={cty}GGEXPCAPTCN_X   + .01*{cty}NYGDPMKTPCN')



In [ ]:
#solve the model.
tempdf = themodel(fpol_ExpInv,silent=1,keep=f'1 % of GDP increase in Govt investment spending')
#themodel.lastdf['IDNNECONGOVTCN']/themodel.basedf['IDNNECONGOVTCN']



### Increase in expenditure on Transfers to households

In this scenario the same fiscal effort is implemented as an increase in transfers to households.


In [ ]:
fpol_ExpTrans=bline.copy()
fpol_ExpTrans=themodel.fix(bline,f'{GGexp}') # Freeze spending levels 
fpol_ExpTrans=themodel.fix(fpol_ExpTrans,f'{cty}GGEXPTRNSCN')

fpol_ExpTrans=fpol_ExpTrans.mfcalc(f'<2025 2050> {cty}GGEXPTRNSCN_X ={cty}GGEXPTRNSCN_X   + .01*{cty}NYGDPMKTPCN')



In [ ]:
#solve the model.
tempdf = themodel(fpol_ExpTrans,silent=1,keep=f'1 % of GDP increase in transfers to households')
#themodel.lastdf['IDNNECONGOVTCN']/themodel.basedf['IDNNECONGOVTCN']



## Comparisons of results from the fiscal scenarios

The following charts compare results from the different fiscal simulations. Impacts will differ both in terms of their long-term and short-term impacts.  For example a scenario that increased investment would likely have negative impacts on consumption in the short-run but in the longer run could be expected to have an opposite impact on potential output, GDP and perhaps consumption.  A scenario that concentrated on transfers or consumption might have more of a short-term impact on demand but in the long run would have limited (and potentially negative impacts on output), especially if increased fiscal deficits and debt crowded out private sector investment.  As all World Bank models are customized to the country for which they have been built the extent of these effects can vary across models.


Recall the scenarios that were run by interrogating the keep_solutions dictionary.



In [ ]:
for key,value in themodel.keep_solutions.items():
    print(key)

In [ ]:
scenarios_fiscal ='Baseline|1 % of GDP Indirect tax cut|1 % of GDP direct tax cut|1 % of GDP increase in G&S spending|1 % of GDP increase in Govt investment spending|1 % of GDP increase in transfers to households'

### Real GDP impacts and impacts on main Real GDP expenditure components

The following chart use the `with keep_switch()`  clause to select which scenario results are to be plotted, and then plots them on a series of charts using the `.keep_plot()` method with the `samefig` and `diffpct` options set to True. the first option ensures that all the graphs are arranged in a grid and the second expresses the results as a percent deviation from the results in the first scenario specified in the `keep_switch()` -- in this case the baseline scenario. Note, the included scenarios in the `keep_switch()` command are identified by the text used in the initial keep command and separated by a horizontal line "|".

In [ ]:
themodel.smpl(2024,2030)
fig_expenditure = themodel.plot(f'{cty}NYGDPPOTLKN {cty}NYGDPMKTPKN {cty}NECONPRVTKN {cty}NEGDIFTOTKN {cty}NEEXPGNFSKN {cty}NEIMPGNFSKN',
                           scenarios = scenarios_fiscal, 
                           datatype= 'difpctlevel',
                           samefig=True,
                           legend=True,
                           title="Comparison of fiscal scenarios")

fig_expenditure.show


For Pakistan, the GDP results appear to be consistent with expectations.  

Fiscal expansion of all types boosted demand and GDP in the short run (second graph).  

However, the long run the impact depends on the impact of the spending on potential output.  Sustained increased investment spending served to increase the capital stock and contribute to higher potential and actual GDP.  Spending that focused on household transfers and reduces in indirect taxes has a modest positive impact on potential, mainly reflecting increased labor participation and a modest increase in investment rates in the case of reduced indirect taxes.

Increases in goods and service spending had a broadly neutral impact on potential, while cuts to direct taxation had a fairly strong negative effect as increased disposable income appears to have mainly boosted consumption, while increased debt and higher interest rates served to crowd out investment.

### Impacts on the fiscal accounts

The following command shows the impacts on several of the main fiscal indicators.

In [ ]:
fig_fiscal = themodel.plot(
        f'{cty}NYGDPMKTPCN {cty}GGEXPTOTLCN {cty}GGEXPINTPCN {cty}GGREVTOTLCN {cty}GGBALOVRLCN {cty}GGDBTTOTLCN',
        scenarios = scenarios_fiscal,name='fig_fiscal',
        datatype = 'difpctlevel',samefig=True,legend=True,title="Comparison of fiscal scenarios")
    
fig_fiscal.show

Note that:
* Nominal GDP is increased in all scenarios, mainly reflecting the inflationary impact of the scenario (most scenarios saw real GDP decline).  
* The fiscal account deteriorates in all scenarios as compared with the baseline.
    * Nominal spending increases even in the tax scenarios, but here the driver is increased interest payments as other elements of spending were held constant.
    * Although interest payments as a percent of their initial level are up a lot, the increase as a percent of GDP (see next set of charts is less pronounced).
    * Revenues improve in the spending scenarios because of higher nominal GDP.
    * The fiscal balance deteriorates (becomes more negative) with the extent of the deterioration smallest in the scenarios where real GDP growth is increasing of hit less hard.
    * Public debt is higher in all scenarios

#### Fiscal impacts as a percent of GDP

As observed. higher inflation (due to increased demand in the early years of the simulation) mean that both revenues and expenditures are higher in the simulation scenarios.  

To correct for this effect, the following charts show the results as a percent of GDP.  Here the inflation influences both the numerator and the denominator, so just the net effect is drawn.


In [ ]:
# Uncomment to display the mnemonics and desriptions of the general gobernment (GG) expenditure items 
# (EXP) in nominal local currency (CN)
#themodel[f'{cty}GGEXP*CN'].des  

In [ ]:
gdp_pct_pat = f'{cty}GGBALOVRLCN {cty}GGEXPTOTLCN {cty}GGREVTOTLCN  {cty}GGEXPINTPCN {cty}GGDBTTOTLCN'     
fig_gdp_pct = themodel.plot(
                        pat = gdp_pct_pat,
                        scenarios=scenarios_fiscal, 
                        datatype='difgdppct',
                        name='Comparing fiscal stimulus impacts',
                        title="Comparing fiscal stimulus impacts",
                        start="2024",
                        samefig=True,
                        legend=True)

fig_gdp_pct.show


* Spending as a percent of GDP increases by about 0.8 percent of GDP in the spending scenarios, the result of the original bump up in spending plus increased interest payments as the debt rises on the one hand and higher inflation which increased nominal GDP, stabilizing the spending ratios. 
* Revenues as a percent of GDP are down in the tax reduction scenarios as expected, and down to a lesser extent in the spending scenarios, presumably reflecting a switch in the mix of total expenditure towards categories that are taxed at relatively lower rates.
* Interest payments rise by as much as 0.3 percent of GDP by the end of the period due to higher debt levels and because of higher interest rates as debt to GDP rates rise.
* Higher debt and fiscal borrowing will translate into increased competition for domestic and foreign savings, crowding out private sector investment
* The fiscal balance deteriorates by one and three percent of GDP, with differences reflecting differences in real GDP and inflation impacts, and revenue impacts.
* Debt rises by between 3 and 5 percent of GDP as the permanent increase in fiscal effort accumulates over time.

## Non fiscal simulations

Three non-fiscal scenarios were run.  The first a temporary increase in the monetary policy interest rate, the second a 10 percent depreciation and the final a permanent $20 increase in the price of crude_petrol.

In [ ]:
#themodel['*poly*'].des


### Monetary policy shock

In this shock, it is assumed that the central bank raises its policy rate by 1 percentage point for 1 year.


In [ ]:
Mpol=bline.copy()
Mpol=themodel.fix(bline,f'{GGexp}') # Freeze spending levels 

Mpol=themodel.fix(Mpol,f'{cty}FMLBLPOLYXN',2025,2025) # One year shock to MP but then let the equation take over.
Mpol=Mpol.mfcalc(f'<2025 2025> {cty}FMLBLPOLYXN_X ={cty}FMLBLPOLYXN_X   + 1')


In [ ]:
#solve the model.
tempdf = themodel(Mpol,silent=1,keep=f'1 ppt increase in policy rate in 2025')
#themodel.lastdf['IDNNECONGOVTCN']/themodel.basedf['IDNNECONGOVTCN']

#themodel.lastdf.loc[2020:2030,"IDNFMLBLPOLYFR"]/themodel.basedf.loc[2020:2030,"IDNFMLBLPOLYFR"]

### Exchange rate depreciation

This shock assumes a depreciation of the currency by 10 percent in 2025.


In [ ]:
Mpol_exr=bline.copy()
#Mpol_exr=themodel.fix(bline,f'{GGexp}') # Freeze spending levels 

Mpol_exr=themodel.fix(Mpol,f'{cty}PANUSATLS',2025,2030) # Permanent shock to exchange rate 10% depreciation
Mpol_exr=Mpol_exr.mfcalc(f'<2025 2030> {cty}PANUSATLS_X ={cty}PANUSATLS_X * 1.1')


In [ ]:
#solve the model.
tempdf = themodel(Mpol_exr,silent=1,keep=f'Permanent 10 percent depreciation in 2025')
#themodel.lastdf['IDNNECONGOVTCN']/themodel.basedf['IDNNECONGOVTCN']

#themodel.lastdf.loc[2020:2030,"IDNPANUSATLS"]/themodel.basedf.loc[2020:2030,"IDNPANUSATLS"]

### TFP Shock

This shock explores the effect of a permanent increase in the level of TFP by 1 percent beginning in 2025.


In [ ]:
TFP=bline.copy()
TFP=themodel.fix(bline,f'{GGexp}') # Freeze spending levels 

#TFP=themodel.fix(Mpol,f'{cty}PANUSATLS',2025,2050) # One year shock to MP but then let the equation take over.
TFP=TFP.mfcalc(f'<2025 2050> {cty}NYGDPTFP ={cty}NYGDPTFP * 1.01')


In [ ]:
#solve the model.
tempdf = themodel(TFP,silent=1,keep=f'A permanent 1 percent increase in TFP levels')
#themodel.lastdf['IDNNECONGOVTCN']/themodel.basedf['IDNNECONGOVTCN']

#themodel.lastdf.loc[2020:2030,"IDNNYGDPTFP"]/themodel.basedf.loc[2020:2030,"IDNNYGDPTFP"]

### A permanent 20 percent increase in oil prices

This shock explores the sensitivity of the model to a permanent \$20 increase in global oil and natural gas prices beginning in 2025.  The natural gas price are assumed to rise proportionately to a $20 increase in world crude oil prices.


In [ ]:
themodel['WLDF*GAS*'].des

In [ ]:
Oil=bline.copy()
Oil=themodel.fix(bline,f'{GGexp}') # Freeze spending levels 

# scenario to raise oil prices.  Assume that natural gas prices increase
# by a similar margin as a $20 increase in crude oil prices
Oil=Oil.mfcalc(
    f'<2025 2050> WLDFNHAS_EUR = WLDFNHAS_EUR * (WLDFCRUDE_PETRO+20)/WLDFCRUDE_PETRO')
Oil=Oil.mfcalc(
    f'<2025 2050> WLDFCRUDE_PETRO = WLDFCRUDE_PETRO + 20')


In [ ]:
#solve the model.
tempdf = themodel(Oil,silent=1,keep=f'A permanent 20 increase in global crude-oil prices')
#themodel.lastdf['IDNNECONGOVTCN']/themodel.basedf['IDNNECONGOVTCN']



### Summary  impacts of non-fiscal scenarios

The following graphs show the change in the main macroeconomic indicators (Inflation, proxied here by the inflation rate of household consumption), real GDP, and changes in the current account, fiscal balances and debt levels expressed as a percent of GDP.

 - A string with the names of the the relevant scenarios are created 
 - A plot with the impact vs baseline on inflation (impact on price growth) is created 
 - A plot with the impact vs baseline on GDP in percent is created 
 - A plot with the impact vs baseline on different expenditure items as percent of gdp is created  
 - The tree plots are combined with the | operator 
 

#### Define scenarios 

In [ ]:
scenarios_non_fiscal =  ('Baseline|'
                        '1 ppt increase in policy rate in 2025|'
                        'Permanent 10 percent depreciation in 2025|'
                        'A permanent 1 percent increase in TFP levels|'
                        'A permanent 20 increase in global crude-oil prices')
#scenarios_non_fiscal

#### Create inflation plot

In [ ]:
fig_impact_inflation  = themodel.plot(pat=f'{cty}NECONPRVTXN',
                              datatype='difgrowth',
                              scenarios = scenarios_non_fiscal,
                              ax_title_template = 'Impact inflation',
                              legend=True,
                              name='inflation');
#fig_impact_inflation

#### Create GDP plot

In [ ]:
fig_impact_gdp = themodel.plot(pat=f'{cty}NYGDPMKTPKN',
                              datatype='difpctlevel',
                              scenarios = scenarios_non_fiscal,                              
                              ax_title_template = 'Impact GDP',                              
                              legend=True,
                              name='gdp');
#fig_impact_gdp

#### Create expenditure plot 

In [ ]:
fig_impact_non_fiscal_gdppct = themodel.plot(
                              pat= f'{cty}BNCABFUNDCN {cty}GGBALOVRLCN {cty}GGDBTTOTLCN',
                              datatype='difgdppct',
                              scenarios = scenarios_non_fiscal,
                              ax_title_template = 'Impact {var_description}',
                              samefig=True,
                              legend=True,
                              name='expenditure')
#fig_impact_non_fiscal_gdppct                    

In [ ]:
fig_non_financial  = fig_impact_inflation | fig_impact_gdp | fig_impact_non_fiscal_gdppct
    
fig_non_financial.set_options(samefig=True,ncol=2,size=(13,8)).show      

As might be expected, effects across these scenarios are more divergent.  

**Inflation**
Higher oil prices and an exchange rate depreciation both are inflationary at least in the short-run.  The inflation impact in the oil scenario is much smaller than in the depreciation scenario and dissipates quite rapidly  as the economy adjusts. 

Both the TFP shock and the monetary policy tightening contribute to declines in inflation.  The TFP shock is deflationary because it raises potential output and therefore open up a negative output gap, at least initially.  Over time this effect diminishes. The monetary policy shock which also induces an increase in the output gap (due to lower demand) in the short run that tends to return to its pre-shock inflation rate.

**Real-side effects**
The permanent increase in TFP translates fairly quickly into higher GDP, and GDP continues to rise -- reflecting stronger potential output.  While the initial increase in TFP will have boosted potential the subsequent increase reflects higher investment induced by the increased productivity of capital (and labor).

The oil price, depreciation and monetary policy scenarios cut into spending, depressing demand, but over the longer run demand tends to recover -- though toward a lower level as investment was lower during the weak demand period reducing potential output.

**Current account**
Current account impacts are broadly neutral in the long run, except in the case of the productivity shock where increased competitiveness boosts exports and reduced the current account deficit.  

**Fiscal impacts**
As might be expected, fiscal impacts are smaller in these scenarios, which do not imply explicit changes in fiscal policy.  Revenues tend to follow activity as do expenditures so impacts on debt are modest, except in the monetary policy scenario where higher interest rates contribute to increased debt financing costs and debt levels.



#### Real Gdp impacts of non-fiscal simulations

In [ ]:
fig_impact_real_gdp = themodel.plot(
                              pat= f'{cty}NYGDPPOTLKN {cty}NYGDPMKTPKN {cty}NECONPRVTKN {cty}NEGDIFTOTKN {cty}NEEXPGNFSKN {cty}NEIMPGNFSKN',
                              datatype='difpctlevel',
                              scenarios = scenarios_non_fiscal,
                              ax_title_template = 'Impact {var_description}',
                              samefig=True,
                              legend=True,                              
                              name='expenditure')
fig_impact_real_gdp.show

#### Discussion of real GDP impacts of non-fiscal scenarios

##### The TFP shock (green line)
Potential and real GDP impacts mirror one another, with the TFP shock raising potential output permanently, and with real GDP catching up over-time slowly.  The potential GDP impact rises proportionately over time because the higher output induces additional investment which adds further to potential GDP.  Higher potential and actual GDP translates into increased consumption, exports and imports as the economy adjusts to the higher scale of activity.

##### The monetary policy shock (blue line)
As can be expected a tightening of monetary policy has negative effects on GDP. Higher interest rates and slower growth reduce investment growth which has a modest negative effect on potential output and long-run GDP. Consumer demand mirrors GDP as incomes are reduced modestly as compared with the baseline.  Exports are hurt initially due to high capital costs, but as inflation declines they benefit and by the end of the period the impact is negligible.  Imports are similarly hurt initially but recover most of the losses.  They remain lower than baseline in line with lower domestic demand and GDP.

##### Depreciation (orange line)
The long run effect of the 10 percent depreciation is broadly neutral as impacts on potential are modest. Both exports and import competing firms benefit as domestic firms  become more competitive internationally. The net effect on GDP is neutralized over the long run, as higher import costs translate into lower real incomes and lower consumption -- bringing GDp back into line with potential. 

##### Oil price hike (red line)

A permanent $20 nominal increase in oil prices has modest impacts on long-run GDP, as potential declines only slightly. Higher oil prices reduce disposable icnome, cutting into consumer demand and real imports.




### Fiscal impacts of non-fiscal scenarios



In [ ]:
# Uncomment to display the mnemonics and desriptions of the general gobernment (GG) expenditure items 
# (EXP) in nominal local currency (CN)
#themodel[f'{cty}GGEXP*CN'].des  

In [ ]:
gdp_pct_pat =       f'{cty}GGBALOVRLCN {cty}GGEXPTOTLCN {cty}GGREVTOTLCN  {cty}GGEXPINTPCN {cty}GGDBTTOTLCN'     
fig_gdp_pct_non_fiscal = themodel.plot(
                    scenarios=scenarios_non_fiscal, 
                    pat = gdp_pct_pat,
                    datatype='difgdppct',
                    name='Comparing other  stimulus impacts',
                    title="Comparing other stimulus impacts",
                    samefig=True,
                    legend=True)

fig_gdp_pct_non_fiscal.show

Fiscal impacts tend to be smaller than in the fiscal scenarios as none of the fiscal levers (spending or tax rates) are being shocked directly.

The monetary policy scenario has the biggest negative impact on the deficit reflecting both an associated increase in spending, likely reflecting automatic stabilizers and a relarively large reduction in revenues as sectors of the economy subject to higher tax levels are also more interest-rate sensitive.  

The other scenarios have limited impact on the deficit.

The deperciation scenario has a relatively pronounced impact on revenues, likely reflecting lower imports and lower tarif revenues.

The two (monetary policy and exchange rate) scenarios that most impact the deficit have the largest impact on the debt. 